# Advanced Techniques for Classification Problems

This notebook covers some of the best techniques for creating machine learning models, including feature selection algorithms, ensembles, and pipelines.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine

from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif
from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

## Feature Selection Exercises:

Wine dataset.

In [22]:
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target
df.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline,target
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0,0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0,0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0,0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0,0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0,0


In [23]:
x = df.drop(columns=["target"])
y = df["target"]

### SelectKBest with score func f_classif

In [24]:
best = SelectKBest(score_func=f_classif, k=4)
best.fit(x, y)
best_features = best.transform(x)
print("Original features:", x.columns)
print("Best features:", best.get_feature_names_out(x.columns))

Original features: Index(['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium',
       'total_phenols', 'flavanoids', 'nonflavanoid_phenols',
       'proanthocyanins', 'color_intensity', 'hue',
       'od280/od315_of_diluted_wines', 'proline'],
      dtype='object')
Best features: ['alcohol' 'flavanoids' 'od280/od315_of_diluted_wines' 'proline']


### Recursive Feature Elimination

In [25]:
model = LogisticRegression(max_iter=200)
rfe = RFE(model, n_features_to_select=4)
rfe.fit(x, y)

rfe.get_feature_names_out(x.columns)

array(['alcohol', 'ash', 'flavanoids', 'od280/od315_of_diluted_wines'],
      dtype=object)

### Getting feature importance with ExtraTreesClassifier

In [26]:
model = ExtraTreesClassifier()
model.fit(x, y)

display({x.columns[i]: model.feature_importances_[i] for i in range(x.shape[1])})

{'alcohol': np.float64(0.11775160662147861),
 'malic_acid': np.float64(0.057271788469853924),
 'ash': np.float64(0.02740927756922321),
 'alcalinity_of_ash': np.float64(0.03808290670759567),
 'magnesium': np.float64(0.03299300186101802),
 'total_phenols': np.float64(0.0645789622849155),
 'flavanoids': np.float64(0.11977981223758886),
 'nonflavanoid_phenols': np.float64(0.03119794485536439),
 'proanthocyanins': np.float64(0.03315426722747916),
 'color_intensity': np.float64(0.12188726522438642),
 'hue': np.float64(0.08627494875084295),
 'od280/od315_of_diluted_wines': np.float64(0.11205066012802588),
 'proline': np.float64(0.15756755806222744)}

## Using Pipelines

For this exercise we will use a limited version of Pime Indians Dataset from tatianaesc github.

In [27]:
url = "https://raw.githubusercontent.com/tatianaesc/datascience/main/diabetes.csv"
df = pd.read_csv(url, delimiter=',')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [28]:
X = df.drop(columns=["Outcome"])
y = df["Outcome"]

SCORING = "accuracy"
SEED = 7
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED)
kfold = KFold(10, shuffle=True, random_state=SEED)

### Creating Pipelines

The ideia is to iterate over every possibility of model and transformer and use croos validation with kfold to find the best model for the problem.

In [31]:
np.random.seed(SEED)

voting_base = [
    ("LogisticRegression", LogisticRegression()),
    ("CART", DecisionTreeClassifier()),
    ("SVC", SVC())
]

models = [
    ("LogisticRegression", LogisticRegression()),
    ("KNeighbors", KNeighborsClassifier()),
    ("CART", DecisionTreeClassifier()),
    ("NaiveBayes", GaussianNB()),
    ("SVC", SVC()),
    ("Bagging", BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50)),
    ("RandomForest", RandomForestClassifier()),
    ("ExtraTrees", ExtraTreesClassifier()),
    ("AdaBoost", AdaBoostClassifier()),
    ("GradientBoosting", GradientBoostingClassifier()),
    ("Voting", VotingClassifier(estimators=voting_base))
]

transformers = [
    ("Original", FunctionTransformer()),
    ("StandardScaler", StandardScaler()),
    ("MinMaxScaler", MinMaxScaler())
]

pipelines: list[Pipeline] = []

for model in models:
    for transformer in transformers:
        pipelines.append(Pipeline((transformer, model)))

best_result: tuple[float, Pipeline] = (0.0, Pipeline((None)))

for pipe in pipelines:
    cv_results = cross_val_score(pipe, X_train, y_train, cv=kfold, scoring=SCORING)
    cv_mean = cv_results.mean()
    if cv_mean > best_result[0]:
        best_result = cv_mean, pipe

display(f"Best Pipeline is: {best_result[1].named_steps.keys()} with mean accuracy: {best_result[0]}")

"Best Pipeline is: dict_keys(['Original', 'LogisticRegression']) with mean accuracy: 0.7689846641988366"